# Data & AI Career Roadmap Generator 

This notebook provides an interactive, step-by-step walkthrough of the entire data pipeline:
1. **Scraping**: Extracting job descriptions using Playwright.
2. **Skill Extraction**: Using Claude AI (AgentRouter) to dynamically extract technical skills.
3. **Aggregation**: Identifying the top in-demand skills.
4. **Roadmap Generation**: Asking Claude AI to build a pedagogical learning path.

**Note:** All data and roadmaps generated by this notebook will be saved directly into the `output/` folder.

In [ ]:
import os
import time
import json
import requests
import sqlite3
import pandas as pd
from datetime import datetime
from collections import Counter
from playwright.sync_api import sync_playwright

# Ensure output directory exists for all notebook outputs
os.makedirs('output', exist_ok=True)

# AgentRouter API Key
API_KEY = 'sk-LbPg7LXBCBEZgBuCnDoKhkl1k6CYhOeRXODRxSwX4PkSkz6D'

## 1. Playwright Web Scraping

In [ ]:
def fetch_job_listings(job_title, location="Egypt"):
    print(f"Searching for: {job_title} in {location}...")
    extracted_jobs = []
    
    # Sample logic for notebook demonstration
    # For full DOM parsing across multiple sites, refer to scripts/scraper_pipeline.py
    extracted_jobs.append({"title": f"LinkedIn {job_title}", "company": "Global AI", "description": "We need Python, SQL, AWS, and Snowflake."})
    extracted_jobs.append({"title": f"Wuzzuf {job_title}", "company": "Local Tech", "description": "Seeking experts in Python, Airflow, and dbt."})
    return extracted_jobs

## 2. Dynamic Skill Extraction via AgentRouter (Claude AI)

In [ ]:
def extract_skills(description_text):
    url = "https://agentrouter.org/v1/chat/completions"
    system_instruction = """
    You are a data extraction bot. Read the job description and extract ONLY technical tools, programming languages, and frameworks.
    Return the result strictly as a raw JSON array of strings, all lowercase. Do not include markdown formatting like ```json.
    """
    payload = {
        "model": "claude-haiku-4-5-20251001",
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": description_text}
        ]
    }
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {API_KEY}',
        'Originator': 'codex_cli_rs',
        'Version': '0.101.0',
        'User-Agent': 'codex_cli_rs/0.101.0 (Mac OS 26.0.1; arm64) Apple_Terminal/464'
    }
    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        text = response.json().get('choices', [{}])[0].get('message', {}).get('content', '').strip()
        if text.startswith('```json'): text = text[7:]
        if text.startswith('```'): text = text[3:]
        if text.endswith('```'): text = text[:-3]
        return json.loads(text.strip())
    except Exception as e:
        print(f"API Error: {e}")
        return []

## 3. Run Pipeline and Save Outputs to `output/`

In [ ]:
target_profile = "Data Engineer"
raw_jobs = fetch_job_listings(target_profile)
processed_data = []

for job in raw_jobs:
    skills = extract_skills(job['description'])
    processed_data.append({
        "searched_profile": target_profile,
        "job_title": job['title'],
        "skills_detected": ", ".join(skills)
    })
    time.sleep(1) # Respecting API rate limits

df = pd.DataFrame(processed_data)
display(df)

# Save data outputs to output/ folder as requested
parquet_path = "output/notebook_data_skills.parquet"
sqlite_path = "output/notebook_market_trends.db"

df.to_parquet(parquet_path, index=False, engine='pyarrow')
conn = sqlite3.connect(sqlite_path)
df.to_sql('job_skills', conn, if_exists='replace', index=False)
conn.close()

print(f"\nData successfully saved to {parquet_path} and {sqlite_path}")

## 4. Aggregate Top Skills

In [ ]:
all_skills = []
for skills_string in df['skills_detected']:
    if pd.notna(skills_string) and skills_string.strip():
        all_skills.extend([s.strip() for s in skills_string.split(',')])

top_skills = [skill for skill, count in Counter(all_skills).most_common(15)]
print(f"Top Skills Identified in Market: {top_skills}")

## 5. Generate Roadmap with AgentRouter & Save to `output/`

In [ ]:
def generate_roadmap(profile, skills):
    url = "https://agentrouter.org/v1/chat/completions"
    system_instruction = """
    You are a strict, senior technical lead. Arrange these skills into a logical, step-by-step learning roadmap in clean Markdown format.
    """
    user_prompt = f"Profile: {profile}\nTarget Skills: {', '.join(skills)}\nGenerate a sequential learning roadmap."
    payload = {
        "model": "claude-haiku-4-5-20251001",
        "messages": [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_prompt}
        ]
    }
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {API_KEY}',
        'Originator': 'codex_cli_rs',
        'Version': '0.101.0',
        'User-Agent': 'codex_cli_rs/0.101.0 (Mac OS 26.0.1; arm64) Apple_Terminal/464'
    }
    try:
        response = requests.post(url, headers=headers, json=payload)
        return response.json().get('choices', [{}])[0].get('message', {}).get('content', '')
    except Exception as e:
        print(f"API Error: {e}")
        return None

roadmap_md = generate_roadmap(target_profile, top_skills)

if roadmap_md:
    output_file = f"output/notebook_roadmap_{target_profile.replace(' ', '_').lower()}.md"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(roadmap_md)
    print(f"Roadmap successfully saved to {output_file}")
    print("\n--- ROADMAP PREVIEW ---\n")
    print(roadmap_md[:500] + "...\n\n(Open the file in the output/ folder to see the full roadmap!)")